Extract and process data in different files into intermediate format before combine into single table `dim_meals`

In [ ]:
%cd ../../..

In [ ]:
import re
import sys

import polars as pl
import numpy as np
import pandas as pd
from loguru import logger

In [ ]:
logger.remove()
logger.add(sys.stdout, level="DEBUG")

In [ ]:
cols = ['meal_id', 'names', 'restaurants', 'meal_type',	'schoolyear', 'attributes', 'co2']

# Transform `menus.xlsx`, sheet `meals`

In [ ]:
meals_raw = pd.read_excel("data/inter/menus.xlsx", sheet_name="meals")

meals_raw.head()

In [ ]:
# Remove duplicate entries
meal_list = (
    meals_raw
    .dropna(axis=0, how='any')
    .groupby('meal_code')
    .last()
    .reset_index()
    .rename(columns={
        'meal_code': 'meal_id',
        'meal_name': 'meal',
        'category': 'meal_type',
        'CO2': 'co2',
    })
)


# Decompose meal name and tag
pat1 = r"(\s?\(\s?[A-Z](\w|\,|\s)*\))"
pat2 = r"(\(|\)|\s)"

def _f_extract_tag(s: str):
    meal_name, tag = "", ""

    s = s.strip()
    out = re.findall(pat1, s)
    if len(out) < 1:
        meal_name = s
    else:
        meal_name = re.sub(pat1, "", s).strip()
        tag = out[0][0]
        tag = re.sub(pat2, '', tag)

    return pd.Series({'meal': meal_name, 'tag': tag})
    
meal_list['meal'] = meal_list['meal'].str.strip().apply(_f_extract_tag)['meal']
meal_list = meal_list.groupby('meal').first().reset_index()


meal_list['meal_type'] = meal_list['meal_type'].str.strip().map({
    'kala': 'fish',
    'liha': 'meat',
    'kana': 'chicken',
    'vegaani': 'vegan',
    'kasvis': 'vegetarian',
    'keskiarvo': 'buffet',
})

meal_list['meal_id'] = meal_list['meal_id'].astype(int)

meal_list = meal_list.sort_values('meal_id').groupby('meal').first().reset_index()

meal_list = meal_list[~meal_list['meal'].str.lower().str.contains('take away')]

meal_list['schoolyear'] = "23-24"
meal_list['restaurants'] = None
meal_list['attributes'] = None

meal_list = meal_list.rename(columns={'meal': 'names'})

meal_list[cols].head()

In [ ]:
path = "data/inter/inter_dim_meals/menus_meals.xlsx"
meal_list[cols].to_excel(path, index=False)

# Transform `menus.xlsx`, sheet `week_1` -> `week_6`

In [ ]:
path = "data/inter/menus.xlsx"

weeks = ["week1", "week2", "week3", "week4", "week5", "week6"]

list_menus = []
for week in weeks:
    raw = pd.read_excel(path, sheet_name=week)

    list_menus.append(raw)

menus_raw = pd.concat(list_menus)
menus_raw.head()

In [ ]:
menus = menus_raw.copy()

# Decompose meal name and tag
pat1 = r"(\s?\(\s?[A-Z](\w|\,|\s)*\))"
pat2 = r"(\(|\)|\s)"

def _f_extract_tag(s: str):
    meal_name, tag = "", ""

    s = s.strip()
    out = re.findall(pat1, s)
    if len(out) < 1:
        meal_name = s
    else:
        meal_name = re.sub(pat1, "", s).strip()
        tag = out[0][0]
        tag = re.sub(pat2, '', tag)

    return pd.Series({
        'meal': meal_name,
        'tag': tag
    })
    
menus['meal_name'] = menus['meal_name'].apply(_f_extract_tag)['meal']

# Process meal_id
pat_meal_code = r"(\d*)\s*\/\s*(\d*)"

def _f_extract_meal_id(s):
    match s:
        case int():
            meal_id = s
        case str():
            meal_id = int(s.split('/')[0])

    return meal_id

menus['meal_id'] = menus['meal_id'].apply(_f_extract_meal_id)

# Remove duplicate meals
menus = menus.groupby('meal_id').first().reset_index()

# Row-wise processing 
meal_type_not_phy = ['fish', 'meat', 'today\'s special', 'vegan-kpl', 'vegan-miscellaneous']
meal_type_phy = ['salad', 'baguettes']

def _f_process_row(row) -> pd.Series:
    out = {
        'meal_type': None,
        'schoolyear': None,
        'restaurant': None,
        'attributes': [],
    }

    # schoolyear
    out['schoolyear'] = '24-25'

    # meal_type
    meal_type = row.meal_type.strip()
    if meal_type in ['meat', 'fish']:
        out['meal_type'] = meal_type
    else:
        out['attributes'].append(meal_type)
    if row.is_vegan:
        out['meal_type'] = "vegan"

    # restaurant
    if meal_type in meal_type_not_phy:
        out['restaurant'] = 'not_phy'
    elif meal_type in meal_type_phy:
        out['restaurant'] = 'phy'

    # other attributes
    if row.is_kela:
        out['attributes'].append("kela")
    if row.is_gluten_free:
        out['attributes'].append("gluten_free")

    return pd.Series(out)
processed = menus.apply(_f_process_row, axis=1)
menus = pd.concat(
    [
        menus[['meal_id', 'meal_name']],
        processed
    ],
    axis=1
)

menus['attributes'] = menus['attributes'].apply(lambda x: '|'.join(x))

menus['co2'] = None
menus = menus.rename(columns={
    'meal_name': 'names',
    'restaurant': 'restaurants',
})
menus[cols].head()

In [ ]:
path = "data/inter/inter_dim_meals/menus_week1-6.xlsx"
menus[cols].to_excel(path, index=False)

# Transform `menus.xlsx`, sheet `baguettes`

In [ ]:
baguettes_raw = pd.read_excel("data/inter/menus.xlsx", sheet_name="baguettes")

baguettes_raw.head()

In [ ]:
baguettes = baguettes_raw.copy()

# Rename meal_type
baguettes['meal_type'] = baguettes['meal_type'].map({
    'kasvis': 'vegetarian',
    'kana': 'chicken',
    'vegaani': 'vegan',
    'kala': 'fish',
    'liha': 'meat'
})
baguettes['names'] = baguettes['meal'].str.strip()


# Add other columns
baguettes['restaurants'] = 'phy'

baguettes['schoolyear'] = '24-25'

baguettes['attributes'] = baguettes['is_kela'].apply(lambda x: 'baguettes|kela' if x else 'baguettes')

# Remove redundant columns
baguettes = baguettes.drop(columns=['is_kela', 'meal'])

baguettes[cols].head()

In [ ]:
path = "data/inter/inter_dim_meals/menus_baguettes.xlsx"
baguettes[cols].to_excel(path, index=False)

# Transform `POS`, Jan 2023 -> Oct 2024

In [ ]:
paths = [
    "data/raw/pos/Sold lunches.csv",
    "data/raw/pos/Sold lunches Kumpula 6-8 2024.csv",
    "data/raw/pos/Sold lunches Kumpula 9-10 2024.csv",
    "data/raw/pos/Sold lunches Viikuna 2023.csv",
    "data/raw/pos/Sold lunches Viikuna 2024.csv"
]

cols_name = [
    'date',
    'time',
    'restaurant',
    'meal_type',
    'meal',
    'pcs',
    'co2',
]

raw = []
for path in paths:
    df = pl.read_csv(path, separator=';')
    df.columns = cols_name
    raw.append(df)


pos_raw = pl.concat(raw)
pos_raw.head()

In [ ]:
map_mealtype = {
    'Liha': 'meat',
    'Kala': 'fish',
    'Vegaani': 'vegan',
    'Kasvis': 'vegetarian',
    'Kana': 'chicken',
    'Not Mapped': 'not_mapped'
}
map_restaurant = {
    '600 Chemicum': 'che',
    '610 Physicum': 'phy',
    '620 Exactum':  'exa',
    '570 Viikuna':  'vik',
}
school_year = (
    pl
        .from_dicts([
        {'from': '2022-09-01', 'to': '2023-08-31', 'schoolyear': '22-23'},
        {'from': '2023-09-01', 'to': '2024-08-31', 'schoolyear': '23-24'},
        {'from': '2024-09-01', 'to': '2025-08-31', 'schoolyear': '24-25'},
    ])
    .with_columns(
        pl.col('from').str.to_date(),
        pl.col('to').str.to_date(),
    )
)

pos = (
    pos_raw
    .with_columns(
        pl.col('date').str.to_date("%d.%m.%Y"),
        pl.col('time').str.to_time("%H:%M"),
        pl.col('meal_type').replace(map_mealtype),
        pl.col('meal').str.strip_chars(),
        pl.col('restaurant').replace(map_restaurant),
        pl.col('co2').str.replace(",", ".").str.to_decimal().cast(pl.Float32),
    )


    # Refine column `co2`
    .with_columns(
        (pl.col('co2') / pl.col('pcs')).alias('co2')
    )


    # Add 'schoolyear'
    .join(school_year, how='cross')
    .filter(
        (1 == 1)
        & (pl.col('date') <= pl.col('to'))
        & (pl.col('date') >= pl.col('from'))
    )
    .drop('from', 'to')


    # Get first entry for group
    .group_by('restaurant', 'meal_type', 'meal', 'co2')
    .first()


    # Add other columns and rename current ones
    .with_columns(
        pl.lit(None).alias('meal_id'),
        pl.col('meal').alias('names'),
        pl.col('restaurant').alias('restaurants'),
        pl.lit(None).alias('attributes')
    )

    
    .select(cols)
)

pos.head()

In [ ]:
path = "data/inter/inter_dim_meals/pos_Jan23-Oct24.xlsx"
pos.select(cols).write_excel(path)

# Transform `POS`, Nov 2024 -> March 2025

In [ ]:
paths = [
    "data/raw/pos/Apr_1/Chemicum lounaat ja hiilijalanjäljet 01112024-31032025.csv",
    "data/raw/pos/Apr_1/Exactum lounaat ja hiilijalanjäljet 01112024-31032025.csv",
    "data/raw/pos/Apr_1/Physicum lounaat ja hiilijalanjäljet 01112024-31032025.csv",
    "data/raw/pos/Apr_1/Viikuna lounaat ja hiilijalanjäljet 01112024-31032025.csv",
]

cols_name = [
    'date',
    'time',
    'restaurant',
    'meal_type',
    'meal',
    'pcs',
    'co2',
]

raw = []
for path in paths:
    df = pl.read_csv(path, separator=';')
    df.columns = cols_name
    raw.append(df)


pos_raw = pl.concat(raw)
pos_raw.head()

In [ ]:
map_mealtype = {
    'Liha': 'meat',
    'Kala': 'fish',
    'Vegaani': 'vegan',
    'Kasvis': 'vegetarian',
    'Kana': 'chicken',
    'Not Mapped': 'not_mapped'
}
map_restaurant = {
    '600 Chemicum': 'che',
    '610 Physicum': 'phy',
    '620 Exactum':  'exa',
    '570 Viikuna':  'vik',
}
school_year = (
    pl
        .from_dicts([
        {'from': '2022-09-01', 'to': '2023-08-31', 'schoolyear': '22-23'},
        {'from': '2023-09-01', 'to': '2024-08-31', 'schoolyear': '23-24'},
        {'from': '2024-09-01', 'to': '2025-08-31', 'schoolyear': '24-25'},
    ])
    .with_columns(
        pl.col('from').str.to_date(),
        pl.col('to').str.to_date(),
    )
)
pat_meal_id = r"(\d{1,})"
pat_meal_id_replaced = r"(?:\d{1,})\s"


pos = (
    pos_raw
    .with_columns(
        pl.col('date').str.to_date("%d.%m.%Y"),
        pl.col('time').str.to_time("%H:%M"),
        pl.col('meal_type').replace(map_mealtype),
        pl.col('meal').str.strip_chars().str.replace(pat_meal_id_replaced, ""),
        pl.col('restaurant').replace(map_restaurant),
        pl.col('co2').str.replace(",", ".").str.to_decimal().cast(pl.Float32),
        pl.col('meal').str.extract(pat_meal_id).alias('meal_id').cast(pl.Int32())
    )


    # Refine column `co2`
    .with_columns(
        (pl.col('co2') / pl.col('pcs')).alias('co2')
    )


    # Add 'schoolyear'
    .join(school_year, how='cross')
    .filter(
        (1 == 1)
        & (pl.col('date') <= pl.col('to'))
        & (pl.col('date') >= pl.col('from'))
    )
    .drop('from', 'to')


    # Get first entry for group
    .group_by('restaurant', 'meal_type', 'meal', 'co2')
    .first()


    # Add other columns and rename current ones
    .with_columns(
        pl.col('meal').alias('names'),
        pl.col('restaurant').alias('restaurants'),
        pl.lit(None).alias('attributes')
    )

    
    .select(cols)
)

pos.head()

In [ ]:
path = "data/inter/inter_dim_meals/pos_Nov24-Mar25.xlsx"
pos.select(cols).write_excel(path)